# Step 8 (re-run) — Analisis tren 2015-2025 (Tahap 15-17), LDA K=25 vs BERTopic reduced 25

Menjawab **RQ4** (bagaimana topik berkembang 2015-2025) dan menguji apakah kesimpulan tren bergantung pada metode (**RQ5**).

**Input:** `df_lda_topics_v2.csv` (step5 v2), `df_bertopic_topics_v2.csv` (step6 v2), `topic_alignment_v2.csv` (step7 v2). Opsional: `lda_topic_by_year_counts_v2.csv`, `bertopic_topic_by_year_counts_v2.csv` untuk uji konsistensi.

**Perbedaan dari step8 lama (semua ditetapkan sebelum melihat hasil):**
1. **Proporsi BERTopic memakai penyebut dokumen ber-topik** (outlier dikeluarkan dari penyebut) agar skalanya sebanding dengan LDA; versi penyebut semua dokumen dijalankan sebagai sensitivitas.
2. **Tiga uji tren, bukan satu:** OLS pada proporsi (kontinuitas dengan step8 lama), Kendall tau + Theil-Sen (tidak bergantung asumsi normalitas), dan WLS berbobot jumlah artikel tahunan (tahun 2015 hanya 555 artikel vs 8.536 di 2024, sehingga proporsinya jauh lebih berisik). Kategori hanya disebut **robust** jika ketiganya sepakat.
3. **Koreksi uji berganda** (Benjamini-Hochberg) dilaporkan untuk p-value OLS; ambang p < 0,10 dipertahankan agar konsisten dengan step8 lama.
4. **Uji komposisi sumber:** jumlah jurnal per tahun naik dari 46 (2015) menjadi 265 (2022-2023). Tren topik bisa mencerminkan berubahnya jurnal yang terwakili, bukan perkembangan bidang. Karena itu tren dihitung ulang pada **panel jurnal stabil** (jurnal dengan artikel di >= 8 dari 10 tahun, 2015-2024).
5. **Uji tanpa jurnal tertentu (`LEAVE_OUT_JOURNALS`):** pangsa IJASEIT turun dari 22,7% (2015) ke 3,0% (2024) padahal jumlah artikelnya relatif stabil; topik yang tren-nya berasal dari pengenceran ini bukan bukti perubahan bidang. Panel jurnal stabil belum menangkap efek ini karena IJASEIT terbit setiap tahun. Topik yang kategori atau slope-nya berubah > 50% tanpa jurnal itu ditandai `Journal_mix_sensitive`.
6. **Uji cakupan:** tren LDA dihitung ulang hanya pada dokumen yang paling yakin (cakupan disamakan dengan BERTopic).
7. **2025 tidak dipakai untuk slope** (data parsial, hanya ~43 jurnal, satu jurnal menyumbang > 20%), tetapi tetap ditampilkan.

Kategori (Emerging/Declining/Persistent/Cyclical/New) hanya berlaku sebagai **label deskriptif atas 2015-2024**, bukan prediksi.

In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings('ignore')

# ============ KONFIGURASI (ambang tetap, sama dengan step8 lama) ============
LAST_FULL_YEAR   = 2024
SLOPE_THRESHOLD  = 0.003    # perubahan proporsi per tahun (0,3 poin persen)
CV_THRESHOLD     = 0.5
P_THRESHOLD      = 0.10
NEW_ZERO_YEARS   = 3
MIN_PROP         = 0.001
STABLE_MIN_YEARS = 8        # jurnal harus punya artikel di >= 8 dari 10 tahun (2015-2024)
LEAVE_OUT_JOURNALS = ['IJASEIT']   # uji 'tanpa jurnal ini' (jurnal multidisiplin terbesar; pangsanya turun dari ~23% ke ~3%)
F = dict(lda='df_lda_topics_v2.csv', bt='df_bertopic_topics_v2.csv', align='topic_alignment_v2.csv')
# =============================================================================
missing = [v for v in F.values() if not os.path.exists(v)]
print('File belum ada:', missing); assert not missing
json.dump(dict(leave_out_journals=LEAVE_OUT_JOURNALS, last_full_year=LAST_FULL_YEAR, slope_threshold=SLOPE_THRESHOLD, cv_threshold=CV_THRESHOLD, p_threshold=P_THRESHOLD,
               new_zero_years=NEW_ZERO_YEARS, min_prop=MIN_PROP, stable_min_years=STABLE_MIN_YEARS), open('trend_config_v2.json', 'w'), indent=2)

dl = pd.read_csv(F['lda']); db = pd.read_csv(F['bt'])
assert len(dl) == len(db) and (dl.title_raw.astype(str).values == db.title_raw.astype(str).values).all(), 'Korpus LDA dan BERTopic tidak identik'
d = pd.DataFrame({'tahun': dl.tahun, 'jurnal': dl.jurnal, 'sinta': dl.sinta, 'lda': dl.lda_topic_name, 'lda_prob': dl.topic_prob,
                  'bt': db.bertopic_name, 'bt_id': db.bertopic_id_reduced})
import re
LO_MASK = d.jurnal.str.contains('|'.join(map(re.escape, LEAVE_OUT_JOURNALS)), case=False, na=False)
print(f'Dokumen = {len(d)} | topik LDA = {d.lda.nunique()} | topik BERTopic = {d.loc[d.bt_id != -1, "bt"].nunique()} | outlier BERTopic = {(d.bt_id == -1).sum()}')

## A. Komposisi korpus per tahun (diagnostik validitas tren) dan Figure 1

In [ ]:
comp = d.groupby('tahun').agg(n_articles=('jurnal', 'size'), n_journals=('jurnal', 'nunique'))
comp['articles_per_journal'] = (comp.n_articles / comp.n_journals).round(1)
comp['top_journal_share_pct'] = d.groupby('tahun').jurnal.agg(lambda s: 100 * s.value_counts(normalize=True).iloc[0]).round(1)
comp['leave_out_share_pct'] = (100 * LO_MASK.groupby(d.tahun).mean()).round(1)
comp['bt_outlier_pct'] = (100 * d.groupby('tahun').bt_id.apply(lambda s: (s == -1).mean())).round(1)
comp.to_csv('corpus_composition_by_year_v2.csv'); print(comp.to_string())

fig, ax = plt.subplots(figsize=(8.5, 4.4)); ax2 = ax.twinx()
ax.bar(comp.index.astype(str), comp.n_articles, color='#4C72B0'); ax.set_ylabel('Number of articles'); ax.set_xlabel('Year')
ax2.plot(comp.index.astype(str), comp.n_journals, 'o-', color='#C44E52'); ax2.set_ylabel('Number of journals', color='#C44E52')
ax.get_xticklabels()[-1].set_color('#C44E52'); ax.annotate('partial year', (len(comp) - 1, comp.n_articles.iloc[-1]), xytext=(0, 8), textcoords='offset points', ha='center', fontsize=8, color='#C44E52')
ax.set_title('Figure 1. Articles and journals per year, 2015-2025', fontsize=10); fig.tight_layout()
plt.savefig('figure1_articles_per_year_v2.png', dpi=300, bbox_inches='tight'); plt.show()

## B. Tabel topik x tahun (jumlah dan proporsi)

In [ ]:
def year_tables(df, label_col, denominator='assigned', all_docs=None):
    x = df[df[label_col].notna()]
    counts = pd.crosstab(x.tahun, x[label_col])
    denom = counts.sum(axis=1) if denominator == 'assigned' else all_docs.reindex(counts.index)
    props = counts.div(denom, axis=0)
    return counts, props, denom

n_year = d.groupby('tahun').size()
lda_counts, lda_props, lda_den = year_tables(d, 'lda')
bt_counts, bt_props, bt_den = year_tables(d[d.bt_id != -1], 'bt')                                      # penyebut = dokumen ber-topik
_, bt_props_all, bt_den_all = year_tables(d[d.bt_id != -1], 'bt', denominator='all', all_docs=n_year)   # sensitivitas

# uji konsistensi dengan file step5/step6 (jika ada)
for f, tab in [('lda_topic_by_year_counts_v2.csv', lda_counts), ('bertopic_topic_by_year_counts_v2.csv', bt_counts)]:
    if os.path.exists(f):
        ref = pd.read_csv(f, index_col=0).drop(columns=['Outlier'], errors='ignore')
        same = ref.shape == tab.shape and (ref[tab.columns].values == tab.values).all()
        print(f'Konsistensi dengan {f}:', 'COCOK' if same else 'TIDAK COCOK -> periksa')
print('Total per tahun (LDA):', lda_counts.sum(axis=1).to_dict())
print('Total per tahun (BERTopic, ber-topik):', bt_counts.sum(axis=1).to_dict())

## C. Klasifikasi tren (Tahap 16)
Aturan (identik dengan step8 lama, diterapkan pada 2015-2024): **New/Recent** jika hampir nol di 3 tahun pertama lalu muncul dan slope > 0; **Cyclical** jika CV > 0,5 dan tidak ada tren signifikan; **Emerging** jika slope > +0,003 dan p < 0,10; **Declining** jika slope < -0,003 dan p < 0,10; selain itu **Persistent**. Aturan yang sama dijalankan pada tiga uji (OLS, Kendall/Theil-Sen, WLS).

In [ ]:
def categorize(y, slope, p):
    mean = y.mean(); cv = y.std() / mean if mean > 0 else np.inf
    first_zero = (y[:NEW_ZERO_YEARS] < MIN_PROP).sum() >= NEW_ZERO_YEARS - 1
    later = y[NEW_ZERO_YEARS:].sum() > 0
    if first_zero and later and slope > 0: return 'New/Recent Topic'
    if cv > CV_THRESHOLD and p > P_THRESHOLD: return 'Cyclical/Fluctuating Topic'
    if slope > SLOPE_THRESHOLD and p < P_THRESHOLD: return 'Emerging Topic'
    if slope < -SLOPE_THRESHOLD and p < P_THRESHOLD: return 'Declining Topic'
    return 'Persistent Topic'

def build_trend_table(props, weights):
    ys = props.index.values[props.index.values <= LAST_FULL_YEAR]
    w = weights.loc[ys].values.astype(float); rows = []
    for t in props.columns:
        y = props.loc[ys, t].values.astype(float)
        ols = stats.linregress(ys, y); kt = stats.kendalltau(ys, y); ts = stats.theilslopes(y, ys)
        wl = sm.WLS(y, sm.add_constant(ys.astype(float)), weights=w).fit()
        c1, c2, c3 = categorize(y, ols.slope, ols.pvalue), categorize(y, ts[0], kt.pvalue), categorize(y, wl.params[1], wl.pvalues[1])
        present = ys[y >= MIN_PROP]
        rows.append(dict(Topic=t, Category=c1, Slope=ols.slope, p_value=ols.pvalue, Mean_Proportion=y.mean(), CV=(y.std() / y.mean() if y.mean() > 0 else np.nan),
                         Kendall_tau=kt.statistic, Kendall_p=kt.pvalue, TheilSen_slope=ts[0], Category_Kendall=c2,
                         WLS_slope=wl.params[1], WLS_p=wl.pvalues[1], Category_WLS=c3, Robust=(c1 == c2 == c3),
                         First_year_present=int(present.min()) if len(present) else np.nan, Peak_year=int(ys[y.argmax()]), Peak_share=y.max(),
                         Share_2025_info=props.loc[2025, t] if 2025 in props.index else np.nan))
    T = pd.DataFrame(rows); T['q_value_BH'] = multipletests(T.p_value, method='fdr_bh')[1]
    return T.sort_values('Slope', ascending=False).reset_index(drop=True)

lda_trend = build_trend_table(lda_props, lda_den)
bt_trend = build_trend_table(bt_props, bt_den)
lda_trend.to_csv('lda_trend_classification_v2.csv', index=False); bt_trend.to_csv('bertopic_trend_classification_v2.csv', index=False)
show = ['Topic', 'Category', 'Slope', 'p_value', 'q_value_BH', 'Mean_Proportion', 'Category_Kendall', 'Category_WLS', 'Robust', 'Peak_year']
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 55)
print('=== LDA (K=25) ==='); print(lda_trend[show].round(4).to_string(index=False))
print('\n=== BERTopic reduced (25) ==='); print(bt_trend[show].round(4).to_string(index=False))
for nm, T in [('LDA', lda_trend), ('BERTopic', bt_trend)]:
    print(f'\n{nm}: distribusi kategori ->', T.Category.value_counts().to_dict(), '| robust:', int(T.Robust.sum()), 'dari', len(T))

## D. Robustness tren (Tahap 19)
Kategori dihitung ulang pada: (1) **panel jurnal stabil**, (2) **LDA cakupan disamakan** (hanya dokumen paling yakin), (3) **BERTopic dengan penyebut semua dokumen**. Yang dilaporkan: berapa topik yang tetap pada kategori yang sama dan arah slope yang sama.

In [ ]:
jy = d[d.tahun <= LAST_FULL_YEAR].groupby('jurnal').tahun.nunique()
stable = set(jy[jy >= STABLE_MIN_YEARS].index); ds = d[d.jurnal.isin(stable)]
print(f'Panel jurnal stabil: {len(stable)} jurnal, {len(ds)} artikel ({100*len(ds)/len(d):.1f}% korpus); artikel per tahun: {ds.groupby("tahun").size().to_dict()}')
thr = np.quantile(d.lda_prob, 1 - (d.bt_id != -1).mean()); dc = d[d.lda_prob >= thr]
print(f'LDA cakupan-disamakan: prob >= {thr:.3f} -> {len(dc)} artikel')

variants = {}
_, p, w = year_tables(ds, 'lda'); variants[('LDA', 'stable_journals')] = build_trend_table(p, w)
_, p, w = year_tables(dc, 'lda'); variants[('LDA', 'coverage_matched')] = build_trend_table(p, w)
_, p, w = year_tables(ds[ds.bt_id != -1], 'bt'); variants[('BERTopic', 'stable_journals')] = build_trend_table(p, w)
variants[('BERTopic', 'all_docs_denominator')] = build_trend_table(bt_props_all, bt_den_all)

dlo = d[~LO_MASK]
_, p, w = year_tables(dlo, 'lda'); variants[('LDA', 'leave_out_journals')] = build_trend_table(p, w)
_, p, w = year_tables(dlo[dlo.bt_id != -1], 'bt'); variants[('BERTopic', 'leave_out_journals')] = build_trend_table(p, w)
print(f'Tanpa {LEAVE_OUT_JOURNALS}: {len(dlo)} artikel tersisa ({len(d)-len(dlo)} dikeluarkan)')

# tandai topik yang bergantung pada komposisi jurnal (dan simpan ulang tabel utama dengan kolom tambahan)
for m, T in [('LDA', lda_trend), ('BERTopic', bt_trend)]:
    lo = variants[(m, 'leave_out_journals')].set_index('Topic')
    T['Slope_leave_out'] = T.Topic.map(lo.Slope); T['Category_leave_out'] = T.Topic.map(lo.Category)
    T['Journal_mix_sensitive'] = (T.Category != T.Category_leave_out) | (((T.Slope - T.Slope_leave_out).abs() > 0.5 * T.Slope.abs()) & (T.Slope.abs() >= SLOPE_THRESHOLD))
lda_trend.to_csv('lda_trend_classification_v2.csv', index=False); bt_trend.to_csv('bertopic_trend_classification_v2.csv', index=False)
for m, T in [('LDA', lda_trend), ('BERTopic', bt_trend)]:
    print(f'\n{m}: topik yang bergantung pada komposisi jurnal (tanpa {LEAVE_OUT_JOURNALS}):')
    print(T[T.Journal_mix_sensitive][['Topic', 'Category', 'Slope', 'Category_leave_out', 'Slope_leave_out']].round(4).to_string(index=False))

base = {'LDA': lda_trend.set_index('Topic'), 'BERTopic': bt_trend.set_index('Topic')}
rows = []
for (m, v), T in variants.items():
    T = T.set_index('Topic'); B = base[m]
    for t in B.index:
        if t in T.index:
            rows.append(dict(method=m, variant=v, Topic=t, base_category=B.loc[t, 'Category'], variant_category=T.loc[t, 'Category'],
                             base_slope=B.loc[t, 'Slope'], variant_slope=T.loc[t, 'Slope'], same_category=B.loc[t, 'Category'] == T.loc[t, 'Category'],
                             same_direction=np.sign(B.loc[t, 'Slope']) == np.sign(T.loc[t, 'Slope'])))
RB = pd.DataFrame(rows); RB.to_csv('trend_robustness_v2.csv', index=False)
summ = RB.groupby(['method', 'variant']).agg(n_topics=('Topic', 'size'), pct_same_category=('same_category', lambda s: 100 * s.mean()),
                                            pct_same_direction=('same_direction', lambda s: 100 * s.mean())).round(1)
print(summ.to_string()); summ.to_csv('trend_robustness_summary_v2.csv')
print('\nTopik yang berubah kategori pada panel jurnal stabil:')
print(RB[(RB.variant == 'stable_journals') & (~RB.same_category)][['method', 'Topic', 'base_category', 'variant_category', 'base_slope', 'variant_slope']].round(4).to_string(index=False))

## E. Konsentrasi jurnal per topik (indikator topik yang mencerminkan sumber, bukan bidang)

In [ ]:
def concentration(df, col, exclude=None):
    x = df if exclude is None else df[df[col] != exclude]
    def f(g):
        vc = g.jurnal.value_counts(normalize=True)
        return pd.Series({'n_docs': len(g), 'n_journals': g.jurnal.nunique(), 'top_journal': vc.index[0], 'top_journal_share_pct': 100 * vc.iloc[0]})
    return x.groupby(col).apply(f).sort_values('top_journal_share_pct', ascending=False)
cl, cb = concentration(d, 'lda'), concentration(d[d.bt_id != -1], 'bt')
cl.assign(method='LDA').reset_index().rename(columns={'lda': 'Topic'}).to_csv('topic_journal_concentration_lda_v2.csv', index=False)
cb.assign(method='BERTopic').reset_index().rename(columns={'bt': 'Topic'}).to_csv('topic_journal_concentration_bertopic_v2.csv', index=False)
pd.set_option('display.max_colwidth', 45)
print('LDA - 5 topik dengan konsentrasi jurnal tertinggi:'); print(cl.head(5).round(1).to_string())
print('\nBERTopic - 5 topik dengan konsentrasi jurnal tertinggi:'); print(cb.head(5).round(1).to_string())

## F. Figure 5 dan 6 — prevalensi topik (small multiples, warna = kategori)

In [ ]:
CAT_COL = {'Emerging Topic': '#2CA02C', 'Declining Topic': '#D62728', 'Persistent Topic': '#7F7F7F', 'Cyclical/Fluctuating Topic': '#FF7F0E', 'New/Recent Topic': '#1F77B4'}
def small_multiples(props, table, title, fname, ncols=5):
    order = table.Topic.tolist(); nrows = int(np.ceil(len(order) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 2.3 * nrows), sharex=True); axes = axes.ravel()
    for ax, t in zip(axes, order):
        r = table[table.Topic == t].iloc[0]
        ax.plot(props.index, props[t], '-o', ms=2.5, lw=1.2, color=CAT_COL[r.Category]); ax.axvspan(2024.5, 2025.5, color='grey', alpha=.15)
        ax.set_title(f'{t[:44]}\nslope {r.Slope*100:+.2f} pp/yr', fontsize=7); ax.tick_params(labelsize=6); ax.grid(alpha=.25)
        ax.set_xticks([2015, 2020, 2025])
    for ax in axes[len(order):]: ax.axis('off')
    handles = [plt.Line2D([0], [0], color=c, lw=2, label=k.replace(' Topic', '')) for k, c in CAT_COL.items()]
    fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=8, frameon=False)
    fig.suptitle(title, fontsize=11); fig.tight_layout(rect=[0, 0.04, 1, 0.97]); plt.savefig(fname, dpi=250, bbox_inches='tight'); plt.show()
small_multiples(lda_props, lda_trend, 'Figure 5. LDA topic prevalence (share of articles per year); shaded = 2025 (partial, not used for slope)', 'figure5_lda_topic_prevalence_v2.png')
small_multiples(bt_props, bt_trend, 'Figure 6. BERTopic topic prevalence (share of topic-assigned articles per year)', 'figure6_bertopic_topic_prevalence_v2.png')

## G. Figure 7 — tren pasangan topik LDA-BERTopic (pasangan kuat dari step7) dan kesepakatan tren antar-metode

In [ ]:
AL = pd.read_csv(F['align']); P = AL[AL.strong_match].copy()
ys = lda_props.index.values[lda_props.index.values <= LAST_FULL_YEAR]
rows = []
for r in P.itertuples():
    if r.lda_name in lda_props.columns and r.bt_name in bt_props.columns:
        a, b = lda_props.loc[ys, r.lda_name].values, bt_props.loc[ys, r.bt_name].values
        rows.append(dict(lda_topic=r.lda_name, bt_topic=r.bt_name, lda_category=lda_trend.set_index('Topic').loc[r.lda_name, 'Category'],
                         bt_category=bt_trend.set_index('Topic').loc[r.bt_name, 'Category'], lda_slope=lda_trend.set_index('Topic').loc[r.lda_name, 'Slope'],
                         bt_slope=bt_trend.set_index('Topic').loc[r.bt_name, 'Slope'], spearman_rho=stats.spearmanr(a, b).statistic))
PC = pd.DataFrame(rows); PC['same_category'] = PC.lda_category == PC.bt_category; PC['same_direction'] = np.sign(PC.lda_slope) == np.sign(PC.bt_slope)
PC.to_csv('trend_pair_comparison_v2.csv', index=False)
print(PC.round(4).to_string(index=False))
print(f'\nPasangan kuat: {len(PC)} | arah slope sama: {int(PC.same_direction.sum())} | kategori sama: {int(PC.same_category.sum())} | rho median: {PC.spearman_rho.median():.2f}')

n = len(PC); ncols = 4; nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.4 * ncols, 2.6 * nrows), sharex=True); axes = np.atleast_1d(axes).ravel()
for ax, r in zip(axes, PC.itertuples()):
    ax.plot(lda_props.index, lda_props[r.lda_topic], '-o', ms=2.5, lw=1.2, color='#4C72B0', label='LDA')
    ax.plot(bt_props.index, bt_props[r.bt_topic], '--s', ms=2.5, lw=1.2, color='#DD8452', label='BERTopic')
    ax.axvspan(2024.5, 2025.5, color='grey', alpha=.15); ax.set_title(f'{r.lda_topic[:34]}\n~ {r.bt_topic[:34]}  (rho={r.spearman_rho:.2f})', fontsize=7)
    ax.tick_params(labelsize=6); ax.grid(alpha=.25); ax.set_xticks([2015, 2020, 2025])
for ax in axes[n:]: ax.axis('off')
axes[0].legend(fontsize=7, frameon=False); fig.suptitle('Figure 7. Prevalence of aligned topic pairs (strong matches only), LDA vs BERTopic', fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.97]); plt.savefig('figure7_aligned_topic_trends_v2.png', dpi=250, bbox_inches='tight'); plt.show()

## H. Ringkasan hasil tren (Tahap 20)

In [ ]:
for nm, T in [('LDA (K=25)', lda_trend), ('BERTopic reduced (25)', bt_trend)]:
    print(f'\n===== {nm} =====')
    for cat in ['Emerging Topic', 'Declining Topic', 'New/Recent Topic', 'Cyclical/Fluctuating Topic', 'Persistent Topic']:
        s = T[T.Category == cat]
        print(f'{cat} ({len(s)}):', '; '.join(f"{r.Topic} [slope {r.Slope*100:+.2f} pp/yr, robust={'ya' if r.Robust else 'tidak'}]" for r in s.itertuples()) if len(s) else '-')

## I. File yang dihasilkan
`trend_config_v2.json`, `corpus_composition_by_year_v2.csv`, `lda_trend_classification_v2.csv`, `bertopic_trend_classification_v2.csv`, `trend_robustness_v2.csv`, `trend_robustness_summary_v2.csv`,
`topic_journal_concentration_lda_v2.csv`, `topic_journal_concentration_bertopic_v2.csv`, `trend_pair_comparison_v2.csv`,
`figure1_articles_per_year_v2.png`, `figure5_lda_topic_prevalence_v2.png`, `figure6_bertopic_topic_prevalence_v2.png`, `figure7_aligned_topic_trends_v2.png`.
(Tabel topik x tahun sudah dihasilkan di step5 dan step6; di sini hanya diuji konsistensinya. Figure 8 sudah ada dari step6.)